<a href="https://colab.research.google.com/github/chirontan/FEM/blob/main/Q4_(a%2Cb%2Cc%2Cd).ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **Solution to Q.4.(a)**

In [2]:
import numpy as np

# =========================================================
# QUESTION 4(a)
# Nonlinear spring element routine
# =========================================================

def spring_element(Ui, Uj, ke, alpha):

    # -----------------------------------------------------
    # 1. Element deformation
    # delta_e = Uj - Ui
    # -----------------------------------------------------

    delta = Uj - Ui

    # -----------------------------------------------------
    # 2. Nonlinear spring force
    # q_e = k_e*delta + alpha*delta^3
    # -----------------------------------------------------

    q = ke*delta + alpha*delta**3

    # -----------------------------------------------------
    # 3. Element internal force vector
    # fint = q[-1, 1]^T
    # -----------------------------------------------------

    fint = q * np.array([-1.0, 1.0])

    # -----------------------------------------------------
    # 4. Tangent stiffness matrix
    # kt = (ke + 3*alpha*delta^2)
    #      [[1,-1],[-1,1]]
    # -----------------------------------------------------

    kt = (ke + 3*alpha*delta**2) * np.array([
        [1.0, -1.0],
        [-1.0,  1.0]
    ])

    return delta, q, fint, kt


# =========================================================
# TEST THE ELEMENT ROUTINE
# =========================================================

# Example values
Ui = 0.0
Uj = 0.5

ke = 100.0
alpha = 1.0


# Call the function
delta, q, fint, kt = spring_element(
    Ui, Uj, ke, alpha
)


# =========================================================
# DISPLAY OUTPUT
# =========================================================

print("QUESTION 4(a) - ELEMENT ROUTINE")
print("--------------------------------")

print("Ui =", Ui)
print("Uj =", Uj)

print("\nElement deformation:")
print("delta =", delta)

print("\nNonlinear spring force:")
print("q =", q)

print("\nElement internal force vector:")
print(fint)

print("\nElement tangent stiffness matrix:")
print(kt)

QUESTION 4(a) - ELEMENT ROUTINE
--------------------------------
Ui = 0.0
Uj = 0.5

Element deformation:
delta = 0.5

Nonlinear spring force:
q = 50.125

Element internal force vector:
[-50.125  50.125]

Element tangent stiffness matrix:
[[ 100.75 -100.75]
 [-100.75  100.75]]


# **Solution to Q.4(b)**

In [3]:
import numpy as np

# =========================================================
# QUESTION 4(b)
# GLOBAL ASSEMBLY
# =========================================================

# ---------------------------------------------------------
# Number of nodes
# ---------------------------------------------------------

n_nodes = 4


# ---------------------------------------------------------
# Element connectivity
# Python uses 0-based numbering
#
# Physical numbering:
# e1 : 1-2
# e2 : 2-3
# e3 : 3-4
# e4 : 2-4
# ---------------------------------------------------------

elements = np.array([
    [0, 1],
    [1, 2],
    [2, 3],
    [1, 3]
])


# ---------------------------------------------------------
# Spring properties
# ---------------------------------------------------------

ke = np.array([
    100.0,
    150.0,
    200.0,
    100.0
])

alpha = np.array([
    1.0,
    2.0,
    1.5,
    1.0
])


# ---------------------------------------------------------
# External force vector
# ---------------------------------------------------------

Fext = np.array([
    0.0,
    10.0,
    20.0,
    40.0
])


# ---------------------------------------------------------
# Current displacement vector
# ---------------------------------------------------------
# For example, at the beginning of Newton iteration

U = np.array([
    0.0,
    0.0,
    0.0,
    0.0
])


# ---------------------------------------------------------
# Initialize global quantities
# ---------------------------------------------------------

Fint = np.zeros(n_nodes)

KT = np.zeros((n_nodes, n_nodes))


# ---------------------------------------------------------
# Element loop
# ---------------------------------------------------------

for e in range(4):

    # Get node numbers
    i = elements[e, 0]
    j = elements[e, 1]

    # Current element displacements
    Ui = U[i]
    Uj = U[j]

    # Element deformation
    delta = Uj - Ui

    # Nonlinear spring force
    q = ke[e]*delta + alpha[e]*delta**3

    # Element internal force vector
    fint_e = q * np.array([
        -1.0,
         1.0
    ])

    # Element tangent stiffness
    kt_e = (
        ke[e]
        + 3.0*alpha[e]*delta**2
    ) * np.array([
        [1.0, -1.0],
        [-1.0,  1.0]
    ])

    # -----------------------------------------------------
    # ASSEMBLY
    # -----------------------------------------------------

    Fint[[i, j]] += fint_e

    KT[np.ix_([i, j], [i, j])] += kt_e


# ---------------------------------------------------------
# Global residual
# ---------------------------------------------------------

R = Fint - Fext


# ---------------------------------------------------------
# Free DOFs
# Node 1 is fixed
# ---------------------------------------------------------

free = [1, 2, 3]

R_free = R[free]

KT_ff = KT[np.ix_(free, free)]


# ---------------------------------------------------------
# OUTPUT
# ---------------------------------------------------------

print("QUESTION 4(b) - GLOBAL ASSEMBLY")
print("================================")

print("\nDisplacement vector:")
print(U)

print("\nGlobal internal force vector:")
print(Fint)

print("\nGlobal external force vector:")
print(Fext)

print("\nGlobal residual:")
print(R)

print("\nFree-DOF residual:")
print(R_free)

print("\nGlobal tangent stiffness matrix:")
print(KT)

print("\nFree-DOF tangent stiffness matrix:")
print(KT_ff)

QUESTION 4(b) - GLOBAL ASSEMBLY

Displacement vector:
[0. 0. 0. 0.]

Global internal force vector:
[0. 0. 0. 0.]

Global external force vector:
[ 0. 10. 20. 40.]

Global residual:
[  0. -10. -20. -40.]

Free-DOF residual:
[-10. -20. -40.]

Global tangent stiffness matrix:
[[ 100. -100.    0.    0.]
 [-100.  350. -150. -100.]
 [   0. -150.  350. -200.]
 [   0. -100. -200.  300.]]

Free-DOF tangent stiffness matrix:
[[ 350. -150. -100.]
 [-150.  350. -200.]
 [-100. -200.  300.]]


# **Solution to Q.4.(c)**

In [4]:
import numpy as np

# =========================================================
# QUESTION 4(c)
# Newton-Raphson solution
# =========================================================

n_nodes = 4

# Element connectivity
elements = np.array([
    [0, 1],   # e1: 1-2
    [1, 2],   # e2: 2-3
    [2, 3],   # e3: 3-4
    [1, 3]    # e4: 2-4
])

# Element properties
ke = np.array([100.0, 150.0, 200.0, 100.0])
alpha = np.array([1.0, 2.0, 1.5, 1.0])

# External forces
Fext = np.array([0.0, 10.0, 20.0, 40.0])

# Node 1 fixed
free = [1, 2, 3]

# Initial displacement
U = np.zeros(4)

# Convergence parameters
tol = 1e-8
max_iter = 50

# Norm of external free force
Fext_norm = np.linalg.norm(Fext[free])

# History
history = []


# ---------------------------------------------------------
# Newton-Raphson loop
# ---------------------------------------------------------

for k in range(max_iter):

    # Global internal force
    Fint = np.zeros(n_nodes)

    # Global tangent stiffness
    KT = np.zeros((n_nodes, n_nodes))

    # -----------------------------------------------------
    # Element loop
    # -----------------------------------------------------

    for e in range(4):

        i = elements[e, 0]
        j = elements[e, 1]

        Ui = U[i]
        Uj = U[j]

        # Element deformation
        delta = Uj - Ui

        # Nonlinear spring force
        q = ke[e]*delta + alpha[e]*delta**3

        # Element internal force
        fint_e = q * np.array([-1.0, 1.0])

        # Element tangent stiffness
        kt_e = (
            ke[e] + 3.0*alpha[e]*delta**2
        ) * np.array([
            [1.0, -1.0],
            [-1.0, 1.0]
        ])

        # Assemble
        Fint[[i, j]] += fint_e

        KT[np.ix_([i, j], [i, j])] += kt_e


    # -----------------------------------------------------
    # Residual
    # -----------------------------------------------------

    R = Fint - Fext

    Rf = R[free]

    residual_ratio = (
        np.linalg.norm(Rf)
        / Fext_norm
    )


    # -----------------------------------------------------
    # Check convergence
    # -----------------------------------------------------

    if residual_ratio < tol:

        history.append([
            k,
            U[1],
            U[2],
            U[3],
            residual_ratio,
            np.nan
        ])

        break


    # -----------------------------------------------------
    # Free-free tangent matrix
    # -----------------------------------------------------

    Kff = KT[np.ix_(free, free)]


    # -----------------------------------------------------
    # Newton correction
    #
    # Kff * dU = -Rf
    # -----------------------------------------------------

    dU = np.linalg.solve(
        Kff,
        -Rf
    )

    correction_norm = np.linalg.norm(dU)


    # -----------------------------------------------------
    # Store history
    # -----------------------------------------------------

    history.append([
        k,
        U[1],
        U[2],
        U[3],
        residual_ratio,
        correction_norm
    ])


    # -----------------------------------------------------
    # Update
    # -----------------------------------------------------

    U[free] += dU


else:

    print("WARNING: Newton-Raphson did not converge.")


# =========================================================
# OUTPUT
# =========================================================

print("\nQUESTION 4(c)")
print("Newton-Raphson Solution")
print("========================")

print("\nFinal displacement vector:")
print(U)

print("\nNewton iteration history:")
print(
    "k       U2          U3          U4"
    "        Residual        Correction"
)

for row in history:

    k = int(row[0])

    if np.isnan(row[5]):

        print(
            f"{k:1d}   "
            f"{row[1]:.9f}  "
            f"{row[2]:.9f}  "
            f"{row[3]:.9f}   "
            f"{row[4]:.5e}       -"
        )

    else:

        print(
            f"{k:1d}   "
            f"{row[1]:.9f}  "
            f"{row[2]:.9f}  "
            f"{row[3]:.9f}   "
            f"{row[4]:.5e}   "
            f"{row[5]:.5e}"
        )


QUESTION 4(c)
Newton-Raphson Solution

Final displacement vector:
[0.         0.69661945 0.91184736 0.97336624]

Newton iteration history:
k       U2          U3          U4        Residual        Correction
0   0.000000000  0.000000000  0.000000000   1.00000e+00   1.51073e+00
1   0.700000000  0.915384615  0.976923077   6.61610e-03   6.04867e-03
2   0.696619691  0.911847595  0.973366477   5.21530e-07   4.09260e-07
3   0.696619454  0.911847359  0.973366241   2.73549e-15       -


# **Solution to Q.4.(d)**

In [5]:
# =========================================================
# QUESTION 4(d)
# FINAL RESULTS AND EQUILIBRIUM VERIFICATION
# =========================================================

print("\n")
print("QUESTION 4(d)")
print("=============")


# =========================================================
# FINAL CONVERGED DISPLACEMENTS
# =========================================================

print("\nFinal converged nodal displacements:")
print("------------------------------------")

print(f"U1 = {U[0]:.10f}")
print(f"U2 = {U[1]:.10f}")
print(f"U3 = {U[2]:.10f}")
print(f"U4 = {U[3]:.10f}")


# =========================================================
# ELEMENT DEFORMATIONS AND FORCES
# =========================================================

print("\nElement results:")
print("---------------------------------------------")
print("Element    Nodes       delta_e          q_e")
print("---------------------------------------------")

element_forces = []
element_deformations = []

for e in range(4):

    i = elements[e, 0]
    j = elements[e, 1]

    # Element deformation
    delta = U[j] - U[i]

    # Nonlinear element force
    q = ke[e] * delta + alpha[e] * delta**3

    element_deformations.append(delta)
    element_forces.append(q)

    print(
        f"{e+1:3d}       "
        f"{i+1}-{j+1}       "
        f"{delta: .10f}    "
        f"{q: .10f}"
    )


# =========================================================
# REASSEMBLE FINAL INTERNAL FORCE VECTOR
# =========================================================

Fint_final = np.zeros(n_nodes)

for e in range(4):

    i = elements[e, 0]
    j = elements[e, 1]

    delta = U[j] - U[i]

    q = ke[e] * delta + alpha[e] * delta**3

    fint_e = q * np.array([-1.0, 1.0])

    # Assemble
    Fint_final[[i, j]] += fint_e


# =========================================================
# FINAL RESIDUAL
# =========================================================

R_final = Fint_final - Fext


print("\nFinal internal force vector:")
print(Fint_final)

print("\nFinal external force vector:")
print(Fext)

print("\nFinal residual vector:")
print(R_final)


# =========================================================
# 4(d)(i)
# EQUILIBRIUM AT THE CONVERGED FREE NODES
# =========================================================

print("\n")
print("4(d)(i) EQUILIBRIUM AT CONVERGED FREE NODES")
print("============================================")

# Free nodes = nodes 2, 3 and 4
free = [1, 2, 3]

R_free_final = R_final[free]

print("\nResidual at free nodes:")
print(R_free_final)

print("\nNorm of free-node residual:")
print(np.linalg.norm(R_free_final))

# Check each free node individually

print("\nFree-node equilibrium check:")

for i in free:

    print(
        f"Node {i+1}: "
        f"R = {R_final[i]:.12e}"
    )

if np.linalg.norm(R_free_final) < 1e-8:

    print("\nRESULT: Free-node equilibrium is satisfied.")

else:

    print("\nRESULT: Free-node equilibrium is NOT satisfied.")


# =========================================================
# SUPPORT REACTION AT NODE 1
# =========================================================

R1 = R_final[0]

print("\nSupport reaction at Node 1:")
print(f"R1 = {R1:.10f} N")


# =========================================================
# 4(d)(ii)
# OVERALL GLOBAL FORCE EQUILIBRIUM
# =========================================================

print("\n")
print("4(d)(ii) OVERALL GLOBAL FORCE EQUILIBRIUM")
print("=========================================")

F2 = Fext[1]
F3 = Fext[2]
F4 = Fext[3]

global_equilibrium = R1 + F2 + F3 + F4

print("\nR1 =", R1)
print("F2 =", F2)
print("F3 =", F3)
print("F4 =", F4)

print("\nChecking:")
print("R1 + F2 + F3 + F4 =")
print(
    f"{R1:.10f} + "
    f"{F2:.10f} + "
    f"{F3:.10f} + "
    f"{F4:.10f}"
)

print(
    f"\nR1 + F2 + F3 + F4 = "
    f"{global_equilibrium:.12e}"
)

if abs(global_equilibrium) < 1e-8:

    print("\nRESULT: Overall global force equilibrium is satisfied.")

else:

    print("\nRESULT: Overall global force equilibrium is NOT satisfied.")



QUESTION 4(d)

Final converged nodal displacements:
------------------------------------
U1 = 0.0000000000
U2 = 0.6966194544
U3 = 0.9118473587
U4 = 0.9733662408

Element results:
---------------------------------------------
Element    Nodes       delta_e          q_e
---------------------------------------------
  1       1-2        0.6966194544     70.0000000000
  2       2-3        0.2152279042     32.3041256622
  3       3-4        0.0615188821     12.3041256622
  4       2-4        0.2767467864     27.6958743378

Final internal force vector:
[-70.  10.  20.  40.]

Final external force vector:
[ 0. 10. 20. 40.]

Final residual vector:
[-7.00000000e+01  1.24344979e-13 -1.42108547e-14  7.10542736e-15]


4(d)(i) EQUILIBRIUM AT CONVERGED FREE NODES

Residual at free nodes:
[ 1.24344979e-13 -1.42108547e-14  7.10542736e-15]

Norm of free-node residual:
1.2535593018283003e-13

Free-node equilibrium check:
Node 2: R = 1.243449787580e-13
Node 3: R = -1.421085471520e-14
Node 4: R = 7.10542